# Deep Research End-to-End Workflow - May 2026 Release

## Overview
This notebook demonstrates the complete end-to-end workflow for deep research analysis, integrating multiple AI agents to provide comprehensive insights and recommendations.

## Solution Architecture

```mermaid
graph TD
    A[Raja - Anomaly<br/>COC_CMN_TRND_MTRC_INSGHT_STG<br/><br/>Top 3 States | Top 5 Providers | Top 5 DRG<br/>Top 5 Provider/DRG within each state]

    B[Rajib - Deep Dive<br/>COC_CMN_DATA_INSGHT_STG<br/>Deepdive & KeyInsight Summary<br/><br/>For each state: DRG, auth approvals, metrics]

    C[Semantic Model<br/>Claim, Member, Auth, Network, PI,<br/>Metric Definition]

    D[Correlation Agent<br/>Additional dimensions<br/>Product, Facility, Diagnosis, Procedure, Modifier<br/><br/>3 States | 5 DRG | 5 Providers]

    E[Pattern Analysis Agent<br/>Aggregates and finds common patterns<br/><br/>Pattern-1<br/>Pattern-N]

    F[Reimbursement Policy Agent - SME<br/><br/>Reimbursement Policy<br/>Reimbursement Policy]

    G[Recommendation Agent<br/>Aggregates all patterns and explanations<br/>to generate recommendations]

    H[Final Recommendations]

    A --> B
    B --> C
    B --> D
    D --> E
    E --> F
    E --> G
    F --> G
    G --> H

    style A fill:#e1f5ff,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style B fill:#e1f5ff,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style C fill:#e8eef9,stroke:#173b7a,stroke-width:2px,color:#173b7a
    style D fill:#ffffff,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style E fill:#fff4e1,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style F fill:#fff4e1,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style G fill:#e8f5e9,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style H fill:#e8f5e9,stroke:#173b7a,stroke-width:2px,color:#173b7a
```

## Workflow Steps

1. **Data Loading**: Load insights from Rajib's anomaly detection (KEY_INSIGHT and DEEP_DIVE)
2. **Pattern Analysis**: Extract actionable patterns from DEEP_DIVE reports
3. **Correlation Analysis**: Find additional dimensions (procedures, modifiers) driving each pattern
4. **Policy Extraction**: Retrieve relevant reimbursement policies for identified codes
5. **Recommendation Synthesis**: Aggregate findings to generate actionable recommendations

## API Endpoints
- Pattern Agent: `POST http://localhost:8000/agents/pattern_agent`
- Correlation Agent: `POST http://localhost:8000/agents/correlation`
- Reimbursement Policy Agent: `POST http://localhost:8000/agents/reimbursement_policy`
- Recommendation Agent: `POST http://localhost:8000/agents/recommendation_synthesis`

# Setup & Initialization

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import copy
import pandas as pd
import json
import requests
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")

Python: /Users/AH45807/project/idiscovery-deep-research/.venv/bin/python
Version: 3.13.5 (main, Jun 11 2025, 15:36:57) [Clang 17.0.0 (clang-1700.0.13.3)]


In [3]:
# Load environment variables
project_root = Path.cwd().parent.parent
env_path = project_root / ".env"
load_dotenv(env_path)

print(f"Project root: {project_root}")
print(f"Environment loaded: {env_path.exists()}")

Project root: /Users/AH45807/project/idiscovery-deep-research
Environment loaded: True


In [4]:
from deep_research_utils import SnowparkHelper, EHAPBase
from deep_research_utils.app_constant import AppConstants
from langchain_openai import ChatOpenAI

2026-05-18 11:34:05,671 - policy_extractor.system - INFO - === Policy Extractor Logging Initialized ===
2026-05-18 11:34:05,672 - policy_extractor.system - INFO - Log directory: /Users/AH45807/project/idiscovery-deep-research/notebooks/examples/logs
2026-05-18 11:34:05,672 - policy_extractor.system - INFO - Max file size: 50.0MB
2026-05-18 11:34:05,672 - policy_extractor.system - INFO - Backup count: 10
2026-05-18 11:34:05,673 - policy_extractor.system - INFO - Console output enabled: True
2026-05-18 11:34:05,674 - policy_extractor.system - INFO - Console log level: INFO
2026-05-18 11:34:05,674 - policy_extractor.system - INFO - Console stream: stdout
2026-05-18 11:34:05,675 - policy_extractor.system - INFO - Process ID: 71125
2026-05-18 11:34:05,675 - policy_extractor.system - INFO - Component log levels:
2026-05-18 11:34:05,675 - policy_extractor.system - INFO -   policy_extractor.snowflake_store: WARNING
2026-05-18 11:34:05,676 - policy_extractor.system - INFO -   policy_extractor.s

In [5]:
# Initialize Snowpark connection
snowpark_programmatic_connection_parameters = {
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USER"],
    "password": os.environ["SNOWFLAKE_SECRET"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"]
}

snowpark = SnowparkHelper(
    connection_type="programmatic",
    batch_size=10000,
    max_workers=6,
    enable_metrics=True,
    connection_pool_size=4,
    **snowpark_programmatic_connection_parameters
)

print("✓ Snowpark connection established")

2026-05-18 11:34:09,641 - deep_research_utils.snowflake_helper - INFO - Snowflake session created successfully
2026-05-18 11:34:17,148 - deep_research_utils.snowflake_helper - INFO - Initialized connection pool with 3 additional sessions
✓ Snowpark connection established


In [6]:
# Initialize EHAP authentication
EHAP = EHAPBase(
    base_url=os.environ.get("EHAP_BASE_URL"),
    client_id=os.environ.get("EHAP_CLIENT_ID"),
    client_secret=os.environ.get("EHAP_CLIENT_SECRET"),
    verify=os.environ.get("SSL_CERT_FILE")
)

print("✓ EHAP authentication initialized")

✓ EHAP authentication initialized


In [7]:
# Initialize LLM
llm = ChatOpenAI(
    model=AppConstants.EHAP_LLM_MODEL,
    api_key=EHAP.get_token(),
    extra_body={
        "reasoning_effort": "medium",
        "summary": None
    }
)

print("✓ LLM initialized")

2026-05-18 11:34:17,195 - deep_research_utils.ehap - INFO - Requesting new access token from https://api.horizon.elevancehealth.com/v2/oauth2/token with client_id: BPWOQiwbyghnhWBaBzRYJhL3dU22bbb0
2026-05-18 11:34:17,359 - deep_research_utils.ehap - INFO - Access token generated successfully.
**TOKEN** **TOKEN** **TOKEN** 


/Users/AH45807/project/idiscovery-deep-research/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.horizon.elevancehealth.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✓ LLM initialized


In [8]:
# API Configuration
API_BASE_URL = "http://localhost:8000"
CONVERSATION_ID = f"tutorial_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"API Base URL: {API_BASE_URL}")
print(f"Conversation ID: {CONVERSATION_ID}")

API Base URL: http://localhost:8000
Conversation ID: tutorial_20260518_113417


# Step 1: Load Insights Data

Load the insights generated by Rajib's anomaly detection system. This includes both KEY_INSIGHT and DEEP_DIVE summaries.


This table is stored in Snowflake:
```sql
select * from U01_COC.COC_DTI_STG.coc_cmn_data_insght_stg
```

In [9]:
# Load insights data
# Note: Update this path to match your local environment
insights_csv_path = project_root / "2026-05-14-UAT-data-insight-stg.csv"

# Alternative: You can also query directly from Snowflake if available
# df_insights = snowpark.session.table("COC_CMN_DATA_INSGHT_STG").to_pandas()

df_insights = pd.read_csv(insights_csv_path)
focus_hcc = "IP AUTH" # for 2026-05 release this is the HCC of interest
df_insights = df_insights[df_insights.STATSCL_MDL_CD == focus_hcc]
print(f"Loaded insights shape: {df_insights.shape}")
print(f"\nInsight types:")
print(df_insights.INSGHT_TYPE_NM.value_counts())
print(f"\nColumns: {list(df_insights.columns)}")

Loaded insights shape: (20, 18)

Insight types:
INSGHT_TYPE_NM
DEEP_DIVE      12
KEY_INSIGHT     8
Name: count, dtype: int64

Columns: ['EDL_LOAD_DTM', 'EDL_RUN_ID', 'EDL_SOR_CD', 'KF_TMS', 'EDL_SCRTY_LVL_CD', 'EDL_LOB_CD', 'EDL_EXTRNL_LOAD_CD', 'EDL_CREAT_DTM', 'EDL_INCRMNTL_LOAD_DTM', 'SNAP_YEAR_MNTH_NBR', 'TRND_TM_PRD_END_MNTH_NBR', 'TRND_TM_PRD_CD', 'LOB_CD', 'LOB_SHRT_DESC', 'STATSCL_MDL_CD', 'INSGHT_TYPE_NM', 'JSON_TXT', 'OFSHR_EXCLSN_SOR_CD']


# Step 2: Process First Row - End-to-End Example

We'll walk through the complete workflow for the first row to demonstrate each agent interaction.

In [10]:
for snap_year_mnth_nbr in df_insights.SNAP_YEAR_MNTH_NBR.unique():
    for trnd_tm_prd_end_mnth_nbr in df_insights.TRND_TM_PRD_END_MNTH_NBR.unique():
        for trnd_tm_prd_cd in df_insights.TRND_TM_PRD_CD.unique():
            for lob_shrt_desc in df_insights.LOB_SHRT_DESC.unique():
                for statscl_mdl_cd in df_insights.STATSCL_MDL_CD.unique():
                    print(f"{snap_year_mnth_nbr}, {trnd_tm_prd_end_mnth_nbr}, {trnd_tm_prd_cd}, {lob_shrt_desc}, {statscl_mdl_cd}")
                    break
                break
            break
        break
    break

202604, 202601, R3, Commercial, IP AUTH


In [11]:
first_anomaly = df_insights[(df_insights.SNAP_YEAR_MNTH_NBR == snap_year_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_END_MNTH_NBR == trnd_tm_prd_end_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_CD == trnd_tm_prd_cd) &
                              (df_insights.LOB_SHRT_DESC == lob_shrt_desc) & 
                              (df_insights.STATSCL_MDL_CD == statscl_mdl_cd) &
                              (df_insights.INSGHT_TYPE_NM == 'KEY_INSIGHT')]
anomaly_json = json.loads(json.loads(first_anomaly.JSON_TXT.iloc[0]))
anomaly_json

{'whats_happening': '',
 'top_contributors': {'provider_trends': [{'name': 'EMORY HILLANDALE HOSPITAL',
    'insight': 'Experienced a substantial increase in authorizations',
    'percentage_change': '+490%'},
   {'name': 'EMORY DECATUR HOSPITAL',
    'insight': 'Showed a significant rise in authorization counts',
    'percentage_change': '+239%'},
   {'name': 'YALE NEW HAVEN HOSPITAL',
    'insight': 'Noted a considerable increase in authorizations',
    'percentage_change': '+34%'},
   {'name': "HENRICO DOCTORS' HOSPITAL",
    'insight': 'Saw a moderate increase in authorization counts',
    'percentage_change': '+22%'}],
  'states': [{'name': 'ME',
    'insight': 'Had a significant increase in authorization counts',
    'percentage_change': '+52%'},
   {'name': 'CO',
    'insight': 'Experienced a notable rise in authorization counts',
    'percentage_change': '+14%'}],
  'drgs': [{'name': 'Ungroupable',
    'insight': 'Showed a substantial increase in authorizations',
    'percentag

In [12]:
first_deep_dive = df_insights[(df_insights.SNAP_YEAR_MNTH_NBR == snap_year_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_END_MNTH_NBR == trnd_tm_prd_end_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_CD == trnd_tm_prd_cd) &
                              (df_insights.LOB_SHRT_DESC == lob_shrt_desc) & 
                              (df_insights.STATSCL_MDL_CD == statscl_mdl_cd) &
                              (df_insights.INSGHT_TYPE_NM == 'DEEP_DIVE')]

deep_dive_json = json.loads(json.loads(first_deep_dive.JSON_TXT.iloc[0]))
deep_dive_json

{'report_title': 'IP Authorization Insights - R3 (Snap Month: 202604, Period End: 202601) - Commercial',
 'national_summary': {'description': 'The national total for Commercial IP authorizations R3 is 33,810 (AMFINR). This report covers the top variance drivers by State, DRG, and Provider for the AUTH_CNT metric.'},
 'top_state_drivers': {'section_title': 'TOP STATE DRIVERS',
  'states': [{'state_name': 'CT',
    'overview': 'CT has 1,802 auths out of 33,810 nationally (5.33% of the national total).',
    'medical_necessity_review_mix': 'In CT, of 1,802 authorizations, 95.45% require Medical Necessity review.',
    'service_driver': 'In CT, of total 1,802 authorizations, IP Med/Surg contributes 1,124 authorizations, accounting for 62.38%; IP BH contributes 511 authorizations, accounting for 28.36%; IP OB Dlvry NB contributes 115 authorizations, accounting for 6.38%; NF contributes 55 authorizations, accounting for 3.05%.',
    'authorization_status_mix': 'In CT, of the 1,802 authorizat

## Step 2.1: Correlation Agent/ Waterfall Agent

The Correlation Agent identifies additional dimensions (procedures, modifiers, etc.) that drive the observed patterns. It drill down to find the most significant factors contributing to the pattern.

Start API server:
```bash
source .venv/bin/activate && uvicorn packages.agents.src.agent_api:app --
reload --host 0.0.0.0 --port 8000
```

In [13]:
from datetime import datetime

def get_ecap_start_month(trnd_tm_prd_cd: str, trnd_tm_prd_end_mnth_nbr: int) -> int:
    """
    Compute start month (YYYYMM) for a given ECAP time period.

    Args:
        trnd_tm_prd_cd (int): One of ["R3", "R6", "R12", "YTD"]
        trnd_tm_prd_end_mnth_nbr (int): End month in YYYYMM format

    Returns:
        int: Start month in YYYYMM format
    """

    end_date = datetime.strptime(str(trnd_tm_prd_end_mnth_nbr), "%Y%m")

    def subtract_months(dt, months):
        year = dt.year
        month = dt.month - months

        while month <= 0:
            month += 12
            year -= 1

        return datetime(year, month, 1)

    if trnd_tm_prd_cd.startswith("R"):
        months = int(trnd_tm_prd_cd[1:])
        # standard rolling window (inclusive)
        start_date = subtract_months(end_date, months - 1)

    elif trnd_tm_prd_cd == "YTD":
        start_date = datetime(end_date.year, 1, 1)

    else:
        raise ValueError(f"Unsupported trnd_tm_prd_cd: {trnd_tm_prd_cd}")

    return int(start_date.strftime("%Y%m"))

# Test cases
print(get_ecap_start_month("R3", 202501))   # Expected: 202411
print(get_ecap_start_month("R6", 202501))   # Expected: 202408
print(get_ecap_start_month("R12", 202501))  # Expected: 202402
print(get_ecap_start_month("YTD", 202512))  # Expected: 202501

202411
202408
202402
202501


In [14]:
def convert_current_ecap_time_to_previous_year(current_period_start: int,
                                               current_period_end: int) -> tuple[int, int]:
    """
    Convert ECAP period (YYYYMM) to previous year period.

    Args:
        current_period_start (int): Start period in YYYYMM format
        current_period_end (int): End period in YYYYMM format

    Returns:
        tuple[int, int]: (previous_period_start, previous_period_end)
    """

    def shift_to_previous_year(period: int) -> int:
        year = period // 100
        month = period % 100

        if not (1 <= month <= 12):
            raise ValueError(f"Invalid month in period: {period}")

        return (year - 1) * 100 + month

    previous_period_start = shift_to_previous_year(current_period_start)
    previous_period_end = shift_to_previous_year(current_period_end)

    return previous_period_start, previous_period_end

In [15]:
# if you dont set them here, it will pick up the defaults from the configs/coc_ecap_ip_auth_sematic_view_with_samples.yaml
current_time_period_end = int(trnd_tm_prd_end_mnth_nbr) # critical step to convert the numpy to int64, else API will throw error
current_time_period_start = get_ecap_start_month(trnd_tm_prd_cd, current_time_period_end)
# print("Current period:", current_time_period_start, current_time_period_end)

previous_period_start, previous_period_end = convert_current_ecap_time_to_previous_year(current_time_period_start, current_time_period_end)
# print(f"Previous period: {previous_period_start} to {previous_period_end}")
conversation_id = f"tutorial-{statscl_mdl_cd}-{lob_shrt_desc}-{snap_year_mnth_nbr}-{trnd_tm_prd_cd}-{trnd_tm_prd_end_mnth_nbr}".replace(" ", "_")

correlation_agent_common_payload = {
    "conversation_id": conversation_id,
    "context":{
      "analysis_mode_parameters": {
        "drill_metric": ["expense_detail.total_paid"],
        "period": {
          "rolling_time_dimension": "expense_detail.incurred_month",
          "current_period": {
            "start_time": current_time_period_start,
            "end_time": current_time_period_end
          },
          "previous_period": {
            "start_time": previous_period_start,
            "end_time": previous_period_end
          }
        }
      },
      "filters": [
        {
          "field": "snap_month",
          "operator": "=",
          "value": int(snap_year_mnth_nbr),
          "source": "dimension_match"
        },
        {
          "field": "lob_description",
          "operator": "=",
          "value": lob_shrt_desc,
          "source": "dimension_match"
        }
      ]
    },

  }
# add the HCC filter
if statscl_mdl_cd == "IP AUTH":
  correlation_agent_common_payload["context"]["filters"].append({
        "field": "hcc_high",
        "operator": "=",
        "value": "IP",
        "source": "dimension_match"
      })
else:
    correlation_agent_common_payload["context"]["filters"].append({
        "field": "hcc_medium",
        "operator": "=",
        "value": statscl_mdl_cd,
        "source": "dimension_match"
    })
# correlation_agent_common_payload

In [126]:
%%time
# Call Correlation Agent for all the states 
correlation_agent_url = f"{API_BASE_URL}/agents/correlation"
correlation_results = {
    "states": {},
    "providers": {},
    "drgs": {}
}
for state in anomaly_json["top_contributors"]["states"]:
    correlation_agent_payload = copy.deepcopy(correlation_agent_common_payload)
    correlation_agent_payload["query"] = f"Where did change happen for state {state['name']}? It {state['insight'].lower()} by {state['percentage_change']}"
    correlation_agent_payload["context"]["filters"].append({
        "field": "service_area_state",
        "operator": "=",
        "value": state["name"],
        "source": "dimension_match"
    })
    print(f"Calling Correlation Agent for state '{state['name']}'...")
    correlation_response = requests.post(correlation_agent_url, json=correlation_agent_payload)
    correlation_result = correlation_response.json()

    print(f"Status: {correlation_response.status_code}")
    print(f"Success: {correlation_result.get('status', False)}")
    correlation_results["states"][state["name"]] = correlation_result

# run for all provider_trends
for provider in anomaly_json["top_contributors"]["provider_trends"]:
    correlation_agent_payload = copy.deepcopy(correlation_agent_common_payload)
    correlation_agent_payload["query"] = f"Where did change happen for provider {provider['name']}? It {provider['insight'].lower()} by {provider['percentage_change']}"
    correlation_agent_payload["context"]["filters"].append({
        "field": "rendering_provider_name",
        "operator": "=",
        "value": provider["name"],
        "source": "dimension_match"
    })
    print(f"Calling Correlation Agent for provider '{provider['name']}'...")
    correlation_response = requests.post(correlation_agent_url, json=correlation_agent_payload)
    correlation_result = correlation_response.json()

    print(f"Status: {correlation_response.status_code}")
    print(f"Success: {correlation_result.get('status', False)}")
    correlation_results["providers"][provider["name"]] = correlation_result


#  run for all drgs
for drg in anomaly_json["top_contributors"]["drgs"]:
    correlation_agent_payload = copy.deepcopy(correlation_agent_common_payload)
    correlation_agent_payload["query"] = f"Where did change happen for drg {drg['name']}? It {drg['insight'].lower()} by {drg['percentage_change']}"
    correlation_agent_payload["context"]["filters"].append({
        "field": "drg_name",
        "operator": "=",
        "value": drg["name"],
        "source": "dimension_match"
    })
    print(f"Calling Correlation Agent for drg '{drg['name']}'...")
    correlation_response = requests.post(correlation_agent_url, json=correlation_agent_payload)
    correlation_result = correlation_response.json()

    print(f"Status: {correlation_response.status_code}")
    print(f"Success: {correlation_result.get('status', False)}")
    correlation_results["drgs"][drg["name"]] = correlation_result

Calling Correlation Agent for state 'ME'...
Status: 200
Success: success
Calling Correlation Agent for state 'CO'...
Status: 200
Success: success
Calling Correlation Agent for provider 'EMORY HILLANDALE HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for provider 'EMORY DECATUR HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for provider 'YALE NEW HAVEN HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for provider 'HENRICO DOCTORS' HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Ungroupable'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Chemotherapy without Acute Leukemia as Secondary Diagnosis with MCC'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Tendonitis, Myositis and Bursitis without MCC'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Cesarean Section without Sterilization without CC/MCC'...
Status: 200
Success: success
Calling Cor

In [130]:
fname = f"anomaly{datetime.now().strftime('%Y%m%d')}.json"
print(fname)
with open(fname, 'w') as f:
    json.dump(anomaly_json, f, indent=4)  # indent makes it human-readable

anomaly20260515.json


In [ ]:
fname = f"correlation_results_{datetime.now().strftime('%Y%m%d')}_{conversation_id.replace("tutorial-", "")}.json"
print(fname)
with open(fname, 'w') as f:
    json.dump(correlation_results, f, indent=4)  # indent makes it human-readable

correlation_results_20260515.json


In [17]:
# Call Pattern Analysis Agent
pattern_agent_url = f"{API_BASE_URL}/agents/pattern_agent"
pattern_request = {
    "conversation_id": CONVERSATION_ID,
    "query": "Analyze authorization patterns and extract actionable insights",
    "context": {
        "deep_dive_report": deep_dive_data,
        "correlation_context": correlation_agent_payload_based_on_initial_data
    }
}

print("Calling Pattern Analysis Agent...")
pattern_response = requests.post(pattern_agent_url, json=pattern_request)
pattern_result = pattern_response.json()

print(f"Status: {pattern_response.status_code}")
print(f"Success: {pattern_result.get('status', False)}")

Calling Pattern Analysis Agent...
Status: 200
Success: success


In [18]:
# Extract and display patterns
if pattern_result.get('status') == 'success':
    pattern_output = pattern_result.get('output', {})
    pattern_table = pattern_output.get('patterns', [])
    
    df_patterns = pd.DataFrame(pattern_table)
    print(f"\nExtracted {len(df_patterns)} patterns")
    
    # Display summary of patterns
    for pattern in pattern_table:
        print(f"{pattern['pattern_rank']}. {pattern['business_interpretation']}")
        for obs in pattern["observations"]:
            print(f"\t{obs}")
        filters = pattern["downstream_payload"]["correlation_agent_input"]["context"]["filters"]
        print("##Filters:")
        for filter in filters:
            print(f"\t{filter['field']} {filter['operator']} {filter['value']}")
        print("***************\n")
else:
    print("Pattern analysis failed")
    df_patterns = pd.DataFrame()


Extracted 4 patterns
1. 7,068 OB delivery authorizations include 992 unresolved cases and 4,738 without medical-necessity review, signaling material review-process inefficiency in a high-volume maternity category.
	1. The three OB delivery DRGs total 7,068 authorizations, all within IP OB Dlvry NB.
	2. These DRGs include 992 unresolved cases (519 Partial + 473 No Decision), equal to 14.03% of authorizations.
	3. 4,738 of 7,068 authorizations (67.03%) do not require Medical Necessity review.
	4. 568 approved cases are Administrative review approved across the three DRGs.
##Filters:
	snap_month = 202604
	lob_code = MCM006
	drg_name = Vaginal Delivery without Sterilization or D&C without CC/MCC
	drg_name = Cesarean Section without Sterilization without CC/MCC
	drg_name = Vaginal Delivery without Sterilization or D&C with CC
***************

2. 974 authorizations at Cedars-Sinai and John Peter Smith require medical necessity review, concentrating review workload in two hospitals and eleva

## Step 2.2: Pattern Analysis Agent

The Pattern Analysis Agent extracts meaningful patterns from the DEEP_DIVE report, identifying key trends, outliers, and actionable insights.


Prepared correlation analysis for pattern: OB delivery DRG review friction
Filters: 5


In [ ]:
# Extract and display correlation results
if len(df_patterns) > 0 and correlation_result.get('success'):
    correlation_output = correlation_result.get('result', {}).get('output', {})
    correlation_summary = correlation_output.get('correlation_summary', {})
    top_correlations = correlation_output.get('top_correlations', [])
    
    print(f"\nCorrelation Analysis Summary:")
    print(f"Run ID: {correlation_summary.get('run_id', 'N/A')}")
    print(f"Root Metric: {correlation_summary.get('root_metric', 'N/A')}")
    
    if top_correlations:
        df_correlations = pd.DataFrame(top_correlations)
        display(Markdown("### Top Dimensional Drivers"))
        display(df_correlations[['dimension', 'value', 'contribution_pct', 'delta_value']])
    else:
        print("No correlation data available")
        df_correlations = pd.DataFrame()
else:
    print("Correlation analysis not available")
    df_correlations = pd.DataFrame()

## Step 2.3: Reimbursement Policy Agent

The Reimbursement Policy Agent retrieves relevant medical policies for identified procedure codes and modifiers.

In [ ]:
# Extract procedure codes from correlation results
procedure_codes = []

if len(df_correlations) > 0:
    proc_dims = df_correlations[df_correlations['dimension'].str.contains('procedure', case=False, na=False)]
    procedure_codes = proc_dims['value'].tolist()[:5]  # Limit to top 5

# Fallback: use example codes if no correlation data
if not procedure_codes:
    procedure_codes = ['99291', '99292']  # Example critical care codes

print(f"Procedure codes to analyze: {procedure_codes}")

In [ ]:
# Call Reimbursement Policy Agent
policy_agent_url = f"{API_BASE_URL}/agents/reimbursement_policy"

policy_request = {
    "conversation_id": CONVERSATION_ID,
    "query": f"Extract reimbursement policies for procedures: {', '.join(procedure_codes)}",
    "context": {
        "intent": {
            "raw_question": f"What are the reimbursement policies for {', '.join(procedure_codes)}?",
            "filters": [
                {
                    "field": "procedure_code",
                    "operator": "in",
                    "value": ",".join(procedure_codes),
                    "source": "correlation_analysis"
                }
            ]
        },
        "correlation_summary": correlation_result.get('result', {}).get('output', {}).get('correlation_summary', {}) if 'correlation_result' in locals() else {}
    }
}

print("Calling Reimbursement Policy Agent...")
policy_response = requests.post(policy_agent_url, json=policy_request)
policy_result = policy_response.json()

print(f"Status: {policy_response.status_code}")
print(f"Success: {policy_result.get('success', False)}")

In [ ]:
# Extract and display policy results
if policy_result.get('success'):
    policy_output = policy_result.get('result', {}).get('output', [])
    
    all_policies = []
    for policy_group in policy_output:
        policy_id = policy_group.get('PLCY_ID', 'N/A')
        results = policy_group.get('results', [])
        
        for result in results:
            result['policy_id'] = policy_id
            all_policies.append(result)
    
    df_policies = pd.DataFrame(all_policies)
    print(f"\nExtracted policies for {len(df_policies)} code(s)")
    
    # Display policy summary
    display_cols = ['policy_id', 'code', 'bundling_logic', 'modifier_usage', 'denial_conditions']
    display_cols = [col for col in display_cols if col in df_policies.columns]
    display(df_policies[display_cols])
else:
    print("Policy extraction failed")
    df_policies = pd.DataFrame()

In [ ]:
# Display detailed policy for first code
if len(df_policies) > 0:
    first_policy = df_policies.iloc[0]
    
    display(Markdown(f"### Policy Details for Code: {first_policy['code']}"))
    display(Markdown(f"**Policy ID:** {first_policy['policy_id']}"))
    
    for col in df_policies.columns:
        if col not in ['code', 'policy_id'] and first_policy[col]:
            display(Markdown(f"**{col.replace('_', ' ').title()}:** {first_policy[col]}"))

## Step 2.4: Recommendation Synthesis Agent

The Recommendation Agent synthesizes insights from all previous agents to generate actionable recommendations.

In [ ]:
# Call Recommendation Synthesis Agent
recommendation_agent_url = f"{API_BASE_URL}/agents/recommendation_synthesis"

recommendation_request = {
    "conversation_id": CONVERSATION_ID,
    "query": "Generate actionable recommendations based on pattern, correlation, and policy analysis",
    "context": {
        "patterns": pattern_result.get('result', {}).get('output', {}).get('pattern_table', []) if pattern_result.get('success') else [],
        "correlations": correlation_result.get('result', {}).get('output', {}) if 'correlation_result' in locals() and correlation_result.get('success') else {},
        "policies": policy_result.get('result', {}).get('output', []) if policy_result.get('success') else [],
        "deep_dive_summary": {
            "period": first_row['TRND_TM_PRD_CD'],
            "lob": first_row['LOB_SHRT_DESC'],
            "model": first_row['STATSCL_MDL_CD']
        }
    }
}

print("Calling Recommendation Synthesis Agent...")
recommendation_response = requests.post(recommendation_agent_url, json=recommendation_request)
recommendation_result = recommendation_response.json()

print(f"Status: {recommendation_response.status_code}")
print(f"Success: {recommendation_result.get('success', False)}")

In [ ]:
# Display recommendations
if recommendation_result.get('success'):
    recommendations = recommendation_result.get('result', {}).get('output', {}).get('recommendations', [])
    
    display(Markdown("# Final Recommendations"))
    
    for i, rec in enumerate(recommendations, 1):
        display(Markdown(f"## Recommendation {i}: {rec.get('title', 'N/A')}"))
        display(Markdown(f"**Priority:** {rec.get('priority', 'N/A')}\n"))
        display(Markdown(f"**Description:** {rec.get('description', 'N/A')}\n"))
        
        actions = rec.get('actions', [])
        if actions:
            display(Markdown("**Actions:**"))
            for action in actions:
                display(Markdown(f"- {action}"))
        
        display(Markdown("---"))
else:
    print("Recommendation synthesis not available or failed")

## Summary: First Row Analysis Complete

We have successfully processed the first row through all four agents:
1. ✓ Pattern Analysis: Extracted key patterns
2. ✓ Correlation Analysis: Identified dimensional drivers
3. ✓ Policy Extraction: Retrieved relevant reimbursement policies
4. ✓ Recommendation Synthesis: Generated actionable recommendations

# Step 3: Batch Processing - All Rows

Now we'll process all DEEP_DIVE rows in the dataset using the same workflow.

In [ ]:
def process_insight_row(row, conversation_id):
    """
    Process a single insight row through all agents.
    
    Args:
        row: DataFrame row containing insight data
        conversation_id: Unique conversation identifier
    
    Returns:
        dict: Results from all agents
    """
    results = {
        'row_id': f"{row['SNAP_YEAR_MNTH_NBR']}_{row['TRND_TM_PRD_CD']}_{row['LOB_CD']}_{row['STATSCL_MDL_CD']}",
        'period': row['TRND_TM_PRD_CD'],
        'lob': row['LOB_SHRT_DESC'],
        'model': row['STATSCL_MDL_CD'],
        'success': False,
        'patterns': [],
        'correlations': {},
        'policies': [],
        'recommendations': [],
        'errors': []
    }
    
    try:
        # Parse deep dive data
        deep_dive_data = json.loads(row['JSON_TXT'])
        
        # 1. Pattern Analysis
        pattern_request = {
            "conversation_id": conversation_id,
            "query": "Analyze authorization patterns",
            "context": {"deep_dive_report": deep_dive_data}
        }
        pattern_response = requests.post(f"{API_BASE_URL}/agents/pattern_agent", json=pattern_request)
        
        if pattern_response.status_code == 200:
            pattern_result = pattern_response.json()
            if pattern_result.get('success'):
                results['patterns'] = pattern_result.get('result', {}).get('output', {}).get('pattern_table', [])
        else:
            results['errors'].append(f"Pattern agent failed: {pattern_response.status_code}")
        
        # 2. Correlation Analysis (for first pattern)
        if results['patterns']:
            first_pattern = results['patterns'][0]
            correlation_request = {
                "intent": {
                    "analysis_mode": "cost_change_investigation_over_time_window",
                    "analysis_mode_parameters": {
                        "name": "cost_change_investigation_over_time_window",
                        "drill_metric": ["claims_expense.total_paid"],
                        "explainer_metrics": ["claims_expense.claim_count"],
                        "period": {
                            "rolling_window": ["3_months"],
                            "rolling_time_dimension": "claims_expense.incurred_month",
                            "start_time": int(str(row['SNAP_YEAR_MNTH_NBR'])[:4] + "09"),
                            "end_time": int(str(row['SNAP_YEAR_MNTH_NBR'])[:4] + "11")
                        },
                        "stop_rules": {
                            "max_depth": 3,
                            "top_k_per_level": 3
                        }
                    },
                    "filters": [],
                    "metric_hint": "claims_expense.total_paid",
                    "raw_question": f"Find drivers for {first_pattern.get('pattern_title', '')}"
                },
                "output_root": "/tmp/correlation_runs"
            }
            
            correlation_response = requests.post(f"{API_BASE_URL}/agents/correlation", json=correlation_request)
            if correlation_response.status_code == 200:
                correlation_result = correlation_response.json()
                if correlation_result.get('success'):
                    results['correlations'] = correlation_result.get('result', {}).get('output', {})
            else:
                results['errors'].append(f"Correlation agent failed: {correlation_response.status_code}")
        
        # 3. Policy Extraction
        procedure_codes = ['99291', '99292']  # Default codes
        if results['correlations']:
            # Extract procedure codes from correlation results
            top_correlations = results['correlations'].get('top_correlations', [])
            proc_codes = [c['value'] for c in top_correlations if 'procedure' in c.get('dimension', '').lower()][:3]
            if proc_codes:
                procedure_codes = proc_codes
        
        policy_request = {
            "conversation_id": conversation_id,
            "query": f"Extract policies for {', '.join(procedure_codes)}",
            "context": {
                "intent": {
                    "filters": [{"field": "procedure_code", "operator": "in", "value": ",".join(procedure_codes)}]
                }
            }
        }
        
        policy_response = requests.post(f"{API_BASE_URL}/agents/reimbursement_policy", json=policy_request)
        if policy_response.status_code == 200:
            policy_result = policy_response.json()
            if policy_result.get('success'):
                results['policies'] = policy_result.get('result', {}).get('output', [])
        else:
            results['errors'].append(f"Policy agent failed: {policy_response.status_code}")
        
        # 4. Recommendation Synthesis
        recommendation_request = {
            "conversation_id": conversation_id,
            "query": "Generate recommendations",
            "context": {
                "patterns": results['patterns'],
                "correlations": results['correlations'],
                "policies": results['policies']
            }
        }
        
        recommendation_response = requests.post(f"{API_BASE_URL}/agents/recommendation_synthesis", json=recommendation_request)
        if recommendation_response.status_code == 200:
            recommendation_result = recommendation_response.json()
            if recommendation_result.get('success'):
                results['recommendations'] = recommendation_result.get('result', {}).get('output', {}).get('recommendations', [])
        else:
            results['errors'].append(f"Recommendation agent failed: {recommendation_response.status_code}")
        
        results['success'] = len(results['errors']) == 0
        
    except Exception as e:
        results['errors'].append(f"Processing error: {str(e)}")
    
    return results

print("Batch processing function defined")

In [ ]:
# Process all DEEP_DIVE rows
deep_dive_rows = df_insights[df_insights.INSIGHTS_TYPE == 'DEEP_DIVE']
batch_results = []

print(f"Processing {len(deep_dive_rows)} DEEP_DIVE rows...\n")

for idx, row in deep_dive_rows.iterrows():
    print(f"Processing row {idx + 1}/{len(deep_dive_rows)}: {row['TRND_TM_PRD_CD']} - {row['LOB_SHRT_DESC']} - {row['STATSCL_MDL_CD']}")
    
    batch_conversation_id = f"{CONVERSATION_ID}_batch_{idx}"
    result = process_insight_row(row, batch_conversation_id)
    batch_results.append(result)
    
    print(f"  ✓ Patterns: {len(result['patterns'])}")
    print(f"  ✓ Policies: {len(result['policies'])}")
    print(f"  ✓ Recommendations: {len(result['recommendations'])}")
    if result['errors']:
        print(f"  ⚠ Errors: {len(result['errors'])}")
    print()

print(f"\nBatch processing complete: {len(batch_results)} rows processed")
print(f"Successful: {sum(1 for r in batch_results if r['success'])}")
print(f"With errors: {sum(1 for r in batch_results if not r['success'])}")

In [ ]:
# Create summary DataFrame
batch_summary = pd.DataFrame([{
    'row_id': r['row_id'],
    'period': r['period'],
    'lob': r['lob'],
    'model': r['model'],
    'num_patterns': len(r['patterns']),
    'num_policies': len(r['policies']),
    'num_recommendations': len(r['recommendations']),
    'success': r['success'],
    'errors': ', '.join(r['errors']) if r['errors'] else None
} for r in batch_results])

display(Markdown("## Batch Processing Summary"))
display(batch_summary)

In [ ]:
# Export results to JSON
output_dir = project_root / "output" / "deep_research_results"
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = output_dir / f"batch_results_{timestamp}.json"

with open(output_file, 'w') as f:
    json.dump(batch_results, f, indent=2)

print(f"Results exported to: {output_file}")

# Conclusion

This notebook demonstrated the complete end-to-end workflow for deep research analysis:

1. **Data Loading**: Successfully loaded insights from COC_CMN_DATA_INSGHT_STG
2. **Pattern Analysis**: Extracted actionable patterns from DEEP_DIVE reports
3. **Correlation Analysis**: Identified dimensional drivers (procedures, modifiers)
4. **Policy Extraction**: Retrieved relevant reimbursement policies
5. **Recommendation Synthesis**: Generated actionable business recommendations
6. **Batch Processing**: Processed all rows systematically

## Key Outputs
- Pattern insights with evidence
- Correlation analysis showing dimensional drivers
- Reimbursement policy details
- Prioritized recommendations
- Batch processing results

## Next Steps
- Review recommendations with stakeholders
- Implement high-priority actions
- Monitor impact of changes
- Iterate on patterns and policies

In [22]:
csv_path = "/Users/AH45807/project/idiscovery-deep-research/notebooks/examples/2026_05_05_batch_results.csv"

In [23]:
import pandas as pd
import json
from typing import List, Dict, Any

In [25]:
df = pd.read_csv(csv_path)
df.columns

Index(['SNAP_YEAR_MNTH_NBR', 'TRND_TM_PRD_END_MNTH_NBR', 'TRND_TM_PRD_CD',
       'LOB_CD', 'LOB_SHRT_DESC', 'STATSCL_MDL_CD', 'INSIGHTS_TYPE',
       'JSON_TXT', 'pattern_result', 'correlation_summary',
       'recommendation_result'],
      dtype='str')

In [ ]:
def remove_leading_numbering(arr: List[str]) -> List[str]:
    """
    Removes leading numbering patterns like:
    '1. ', '1) ', '01 - ', etc. from each string.
    """

    pattern = re.compile(r"^\s*\d+\s*[\.\)\-]\s*")

    cleaned = [pattern.sub("", s) for s in arr]
    return cleaned

def format_recommendation(rec_json: list[dict]) -> str:
    rec_str = ""
    i = 1
    for recommendation in rec_json:
        if recommendation["priority"].upper() == "HIGH":
            rec_str += f"{str(i)}. "  + recommendation["description"]
            # print(recommendation["evidence"])
        for ev in remove_leading_numbering(recommendation["evidence"]):
            rec_str += f"\n - {ev}"
        i += 1
        rec_str += "\n\n"
    return rec_str

In [106]:
formatted_outputs = []

for i, row in df.iterrows():
    raw = row['recommendation_result']

    try:
        # Handle JSON stored as string
        rec_json = json.loads(raw.replace("'", "\"")) #if isinstance(raw, str) else raw
        print(i, type(raw))
    except json.JSONDecodeError:
        # If it's already plain text, just use it
        formatted_outputs.append(f"{i+1}. {raw}")
        continue

    formatted_text = format_recommendation(rec_json)
    formatted_outputs.append(formatted_text)
    
    if i == 4:
        break

0 <class 'str'>
1 <class 'str'>
2 <class 'str'>
3 <class 'str'>
5 <class 'str'>
6 <class 'str'>
7 <class 'str'>
9 <class 'str'>
10 <class 'str'>
11 <class 'str'>
12 <class 'str'>
13 <class 'str'>
14 <class 'str'>
15 <class 'str'>
16 <class 'str'>
17 <class 'str'>
18 <class 'str'>
19 <class 'str'>
20 <class 'str'>
21 <class 'str'>
22 <class 'str'>
23 <class 'str'>
24 <class 'str'>
25 <class 'str'>
26 <class 'str'>
27 <class 'str'>
28 <class 'str'>
29 <class 'str'>
30 <class 'str'>
31 <class 'str'>
32 <class 'str'>
33 <class 'str'>
34 <class 'str'>
36 <class 'str'>
37 <class 'str'>
38 <class 'str'>
39 <class 'str'>
40 <class 'str'>
42 <class 'str'>
43 <class 'str'>


In [118]:
print(formatted_outputs[0])

1. Reassess prior authorization policy for Commercial acute-hospital inpatient OB delivery/well newborn services that do not require prior authorization, prioritizing HMO.
 - Paid costs increased $33.6M (7%), almost entirely driven by Commercial acute-hospital inpatient OB delivery/well newborn claims, with 93% from services not requiring prior authorization and 51% in HMO.
 - Three IP OB delivery DRGs total 7,068 authorizations, representing 4.98% of 141,851 national authorizations.
 - These DRGs report 4,738 authorizations that do not require Medical Necessity review (67.03%) and 1,807 that require it (25.57%).


 - Partial and No Decision statuses total 992 cases (14.03%), including 590 in Vaginal Delivery without Sterilization or D&C without CC/MCC and 247 in Vaginal Delivery without Sterilization or D&C with CC.
 - Approved status totals 5,899 cases (83.46%) while Not Approved totals 177 cases (2.50%).
 - These DRGs report 4,738 authorizations that do not require Medical Necessity

In [109]:
rec_json

[{'rank': 1,
  'priority': 'HIGH',
  'category': 'Policy',
  'description': 'Implement targeted utilization management policy for inpatient medical/surgical and unclassified procedures at no-affiliation hospitals in WA Medicare DSNP HMO.',
  'evidence': ['Paid claims rose $401K (+9%), entirely in Medicare DSNP HMO, driven mainly by inpatient medical/surgical spend and unclassified procedures, with no-affiliation hospitals generating 48% of the increase.'],
  'story_alignment': ['Paid claims rose $401K (+9%), entirely in Medicare DSNP HMO.',
   'with no-affiliation hospitals generating 48% of the increase.'],
  'peer_benchmarking': []},
 {'rank': 2,
  'priority': 'MEDIUM',
  'category': 'Policy',
  'description': 'Standardize WA administrative review and partial authorization policy to reduce authorization processing variation.',
  'evidence': ['1. WA has 78 Administrative review approvals out of 298 approved authorizations, or 26.17%.',
   '2. TX records 10.73% Administrative review ap